# 04 · Unseen-drug response prediction

Cell-line representation = HeteroSC fusion of four frozen embeddings; drug representation = edge-gated GCN (or fingerprint MLP);
target = GDSC AUC. Validation = **10-fold leave-drug-out**: each fold's test drugs never appear in training, and early stopping
uses a random 10% of the *training drugs*, so model selection also measures generalisation to unseen drugs.

Needs: `data/drug_response/` (notebook 01) and cell-line embeddings from notebooks `02a–d` Part B.
**Compute:** one fold of one configuration takes roughly 15–25 min per fold on a consumer GPU; a full study
(3 drug encoders + 5 cell encoders + uniform control, 10 folds each) is many GPU-hours. Start with one fold.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from heterosc.data import load_drug_tables, load_cell_line_embeddings, keep_complete_cell_lines
from heterosc.drug_response import (DrugFeatureStore, DrugResponseConfig, assert_leave_drug_out, run_cv, table_fingerprint)
from heterosc.plots import plot_metric_comparison

DATA = Path("data/drug_response")
EMB = Path("outputs/drug_response_embeddings")
EXPERTS = ["geneformer", "scgpt", "cellfm", "scfoundation"]

resp, graphs, fp = load_drug_tables(DATA)
n_folds = assert_leave_drug_out(resp)                       # refuses a table whose folds are not per drug
cell_emb = load_cell_line_embeddings(EMB, EXPERTS)
resp = keep_complete_cell_lines(resp, cell_emb)
store = DrugFeatureStore(graphs, fp)
dims = {e: len(next(iter(cell_emb[e].values()))) for e in EXPERTS}
print(f"{len(resp):,} pairs | {resp.drug_id.nunique()} drugs | {resp.cell_line_id.nunique()} cell lines | {n_folds} folds | fold-table md5 {table_fingerprint(resp)}")

## One fold, HeteroSC + GCN drug encoder

In [ ]:
cfg = DrugResponseConfig(cell_encoder="heterosc", drug_mode="gcn", expert_dims=dims)       # paper settings: 50 epochs, patience 10, lr 1e-4, lambda_H 0.05
res = run_cv(cfg, resp, cell_emb, store, out_dir="outputs/drug_response/heterosc_gcn", folds=[0])
res[["fold", "pcc_per_drug", "pcc_pooled", "mse", "n_test_drugs"]]

## Full cross-validation and baselines
Run the loop below for each configuration. Each finished fold stores the MD5 of the fold table next to its metrics, and a result made with
a *different* table is recomputed instead of silently reused.

In [ ]:
configs = {
    "HeteroSC (adaptive)": DrugResponseConfig(cell_encoder="heterosc", drug_mode="gcn", expert_dims=dims),
    "Uniform fusion":      DrugResponseConfig(cell_encoder="uniform",  drug_mode="gcn", expert_dims=dims),
    **{e: DrugResponseConfig(cell_encoder=e, drug_mode="gcn", expert_dims=dims) for e in EXPERTS},
}
RUN_ALL = False                      # set True for the full study (many GPU-hours)
results = {}
if RUN_ALL:
    for name, c in configs.items():
        results[name] = run_cv(c, resp, cell_emb, store, out_dir=f"outputs/drug_response/{name.split()[0].lower()}_{c.drug_mode}")

In [ ]:
if results:
    plot_metric_comparison({k: v["pcc_per_drug"].values for k, v in results.items()}, "per-drug PCC", title="Leave-drug-out, per fold")
    plt.tight_layout(); plt.show()
    pd.DataFrame({k: v[["pcc_per_drug", "pcc_pooled", "mse"]].mean() for k, v in results.items()}).T.round(4)

Primary metric = **mean per-drug PCC** (Pearson across cell lines for each held-out drug, averaged over drugs with ≥ 3 pairs); pooled PCC
also captures between-drug variation. Pairwise comparisons in the manuscript use matched per-drug PCCs with two-sided Wilcoxon signed-rank tests.